In [2]:
import pandas as pd
import numpy as np
import csv

MERAA_HOURLY = r"C:\DOCUMENTO\Sand-and-Dust-Storms-and-Human-Health\data_prep\merra-2\out_dust_events\hourly_timeseries_with_event_mark.csv"
LANZHOU_CSV  = r"C:\DOCUMENTO\Sand-and-Dust-Storms-and-Human-Health\data_prep\webcrawler\lanzhou_201101_202602.csv"

START = "2021-03-13"
END   = "2021-03-21"
CRIT  = "DUSMASS"  # 你用的关键变量

# ---- MERRA hourly ----
m = pd.read_csv(MERAA_HOURLY)
m["datetime_local"] = pd.to_datetime(m["datetime_local"])
m = m[(m["datetime_local"] >= START) & (m["datetime_local"] < pd.to_datetime(END) + pd.Timedelta(days=1))].copy()

m["DUSMASS_ugm3"] = m[CRIT].astype(float) * 1e9
m["date"] = m["datetime_local"].dt.date

idx = m.groupby("date")["DUSMASS_ugm3"].idxmax()
merra_daily = m.groupby("date").agg(
    merra_max_ugm3=("DUSMASS_ugm3", "max"),
    merra_mean_ugm3=("DUSMASS_ugm3", "mean"),
).reset_index()
merra_peak = m.loc[idx, ["date", "datetime_local", "DUSMASS_ugm3"]].rename(
    columns={"datetime_local":"merra_peak_time_local", "DUSMASS_ugm3":"merra_peak_ugm3"}
)
merra_daily = merra_daily.merge(merra_peak, on="date", how="left")

# ---- Lanzhou daily (自动判断分隔符+中文日期) ----
def sniff_delim(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        s = f.read(4096)
    try:
        return csv.Sniffer().sniff(s, delimiters=[",","\t",";"]).delimiter
    except Exception:
        return "\t" if s.count("\t") > 0 else ","

delim = sniff_delim(LANZHOU_CSV)
try:
    lz = pd.read_csv(LANZHOU_CSV, sep=delim, engine="python")
    # 你的文件可能没有表头；如果有表头，第一列通常叫 date
    if "date" in lz.columns:
        lz["date2"] = pd.to_datetime(lz["date"], errors="coerce")
    else:
        first = lz.columns[0]
        lz["date2"] = pd.to_datetime(lz[first], format="%Y年%m月%d日", errors="coerce")
except Exception:
    lz = pd.read_csv(LANZHOU_CSV, sep=delim, engine="python", header=None)
    lz["date2"] = pd.to_datetime(lz[0], format="%Y年%m月%d日", errors="coerce")

lz = lz.dropna(subset=["date2"]).copy()
lz["date"] = lz["date2"].dt.date
lz = lz[(lz["date2"] >= START) & (lz["date2"] < pd.to_datetime(END) + pd.Timedelta(days=1))].copy()

# 如果你的列名是 aqi_pm10/aqi_pm25/aqi 就能直接看到；否则你自己把列名对应一下
out = lz.merge(merra_daily, on="date", how="left")
cols = [c for c in ["date2","aqi","aqi_pm10","aqi_pm25","merra_max_ugm3","merra_mean_ugm3","merra_peak_time_local","merra_peak_ugm3"] if c in out.columns]
print(out[cols].sort_values("date2").to_string(index=False))


Empty DataFrame
Columns: [date2, aqi, aqi_pm10, aqi_pm25, merra_max_ugm3, merra_mean_ugm3, merra_peak_time_local, merra_peak_ugm3]
Index: []


C:\Users\Cory Kong\AppData\Local\Temp\ipykernel_832\3478614079.py:44: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  lz["date2"] = pd.to_datetime(lz["date"], errors="coerce")


In [5]:
import re
import pandas as pd

def read_lanzhou_csv_robust(path: str) -> pd.DataFrame:
    # 1) 尝试不同编码
    encodings = ["utf-8-sig", "utf-8", "gb18030", "gbk"]
    last_err = None
    df = None
    for enc in encodings:
        try:
            # sep=None + engine=python 会尝试自动识别分隔符（逗号/制表符/分号等）
            df = pd.read_csv(path, sep=None, engine="python", encoding=enc)
            break
        except Exception as e:
            last_err = e
    if df is None:
        raise RuntimeError(f"读取失败，尝试编码 {encodings} 都不行。最后错误：{last_err}")

    # 2) 规范化列名
    df.columns = [str(c).strip() for c in df.columns]

    # 3) 找日期列：优先 date/日期 等，否则用第一列
    cand_cols = ["date", "日期", "Date", "datetime", "time", "时间"]
    date_col = None
    for c in cand_cols:
        if c in df.columns:
            date_col = c
            break
    if date_col is None:
        date_col = df.columns[0]

    s = df[date_col].astype(str).str.strip()

    # 4) 鲁棒解析：先试中文格式，再用正则兜底
    # 4.1 直接按中文格式
    dt = pd.to_datetime(s, format="%Y年%m月%d日", errors="coerce")

    # 4.2 对没解析出来的，用正则抠年月日
    mask = dt.isna()
    if mask.any():
        # 允许 2021-03-14 / 2021/3/14 / 2021年3月14日 / 2021.03.14
        def parse_by_regex(x):
            m = re.search(r"(\d{4})\D+(\d{1,2})\D+(\d{1,2})", x)
            if not m:
                return pd.NaT
            y, mo, d = m.group(1), m.group(2), m.group(3)
            return pd.to_datetime(f"{y}-{int(mo):02d}-{int(d):02d}", errors="coerce")
        dt2 = s[mask].apply(parse_by_regex)
        dt.loc[mask] = dt2

    df["date2"] = dt
    df = df.dropna(subset=["date2"]).copy()
    df["date"] = df["date2"].dt.date

    return df

lz = read_lanzhou_csv_robust(LANZHOU_CSV)

START = "2021-03-13"
END   = "2021-03-21"

lz_sub = lz[(lz["date2"] >= START) & (lz["date2"] <= END)].copy()
print("lz_sub rows:", len(lz_sub))
print(lz_sub[["date2"]].head())

cmp_df = lz_sub.merge(merra_daily, on="date", how="left")
print(cmp_df[["date2","aqi","aqi_pm10","aqi_pm25","merra_max_ugm3","merra_peak_time_local"]].to_string(index=False))


lz_sub rows: 9
          date2
3846 2021-03-13
3847 2021-03-14
3848 2021-03-15
3849 2021-03-16
3850 2021-03-17
     date2   aqi  aqi_pm10  aqi_pm25  merra_max_ugm3 merra_peak_time_local
2021-03-13  78.0     106.0      44.0       32.215701   2021-03-13 00:30:00
2021-03-14 313.0     486.0     117.0      208.476270   2021-03-14 07:30:00
2021-03-15 152.0     254.0      74.0      225.037178   2021-03-15 11:30:00
2021-03-16 500.0    3549.0     657.0      290.057807   2021-03-16 07:30:00
2021-03-17 500.0    2552.0     434.0       94.757915   2021-03-17 23:30:00
2021-03-18 500.0    1564.0     260.0       97.054949   2021-03-18 04:30:00
2021-03-19 422.0     846.0     127.0      262.272782   2021-03-19 23:30:00
2021-03-20 259.0     379.0      58.0      255.428515   2021-03-20 00:30:00
2021-03-21 122.0     194.0      41.0      204.888263   2021-03-21 00:30:00


In [7]:
import os
import re
import stat
import urllib.parse
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr


# ========= 你只要改这里 =========
EARTHDATA_USERNAME = "correr27890"
EARTHDATA_PASSWORD = "AQN/RZ2Y&S5Rb+j"

LINKLIST_PATH = r"C:\Users\Cory Kong\Downloads\subset_M2T1NXAER_5.12.4_20260216_082525_.txt"
DEFAULT_DATASET_DIR = "M2T1NXAER.5.12.4"  # 你用的是这个就别改
HYRAX_BASE = "https://goldsmr4.gesdisc.eosdis.nasa.gov/opendap/hyrax"
# ===============================

# ===== 诊断范围与参数 =====
START = "2021-03-13"
END   = "2021-03-21"

CITY_LAT = 36.0611
CITY_LON = 103.8343
WINDOW_DEG = 1.0

VARS = ["DUSMASS", "DUCMASS", "DUEXTTAU"]  # 你的 linklist 必须勾过这些变量


# ---------- 写 netrc / dodsrc（保证 OPeNDAP 认证稳定） ----------
def ensure_earthdata_auth_files(username: str, password: str):
    home = Path.home()
    if os.name == "nt":
        netrc = home / "_netrc"
        cookies = home / "_urs_cookies"
        dodsrc = home / "_dodsrc"
    else:
        netrc = home / ".netrc"
        cookies = home / ".urs_cookies"
        dodsrc = home / ".dodsrc"

    netrc_content = (
        f"machine urs.earthdata.nasa.gov login {username} password {password}\n"
        f"machine goldsmr4.gesdisc.eosdis.nasa.gov login {username} password {password}\n"
    )
    netrc.write_text(netrc_content, encoding="utf-8")
    if not cookies.exists():
        cookies.write_text("", encoding="utf-8")

    dodsrc.write_text(
        f"HTTP.NETRC={str(netrc)}\nHTTP.COOKIEJAR={str(cookies)}\nHTTP.SSL_VERIFY=0\n",
        encoding="utf-8"
    )
    if os.name != "nt":
        netrc.chmod(stat.S_IRUSR | stat.S_IWUSR)
    return str(netrc), str(cookies), str(dodsrc)


# ---------- 读取 txt：修复“长URL被断行” ----------
def read_urls_repair_wrapped(txt_path: str):
    urls = []
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith("http://") or line.startswith("https://"):
                urls.append(line)
            else:
                if urls:
                    urls[-1] += line
    # 去重保序
    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            out.append(u)
            seen.add(u)
    return out


def infer_original_path_from_sub_filename(sub_name: str, dataset_dir: str):
    m = re.search(r"(MERRA2_\d{3}\.[^.]+\.\d{8})", sub_name)
    if not m:
        return None
    core = m.group(1)
    date = core.split(".")[-1]
    yyyy = date[:4]
    mm = date[4:6]
    return f"/data/MERRA2/{dataset_dir}/{yyyy}/{mm}/{core}.nc4"


def parse_linkslist_to_hyrax_urls(txt_path: str, dataset_dir: str):
    urls = read_urls_repair_wrapped(txt_path)
    hyrax_urls = []

    for u in urls:
        if u.lower().endswith(".pdf"):
            continue

        pu = urllib.parse.urlparse(u)
        qs = urllib.parse.parse_qs(pu.query)

        # case A: HTTP_services has FILENAME=
        if "FILENAME" in qs:
            file_path = urllib.parse.unquote(qs["FILENAME"][0])
            hyrax_urls.append(HYRAX_BASE + file_path)
            continue

        # case B: direct SUB link or nc4 link
        base = os.path.basename(pu.path)

        if "/data/" in pu.path and base.endswith(".nc4"):
            idx = pu.path.find("/data/")
            hyrax_urls.append(HYRAX_BASE + pu.path[idx:])
            continue

        if base.endswith(".SUB.nc") or base.endswith(".SUB.nc4") or base.endswith(".nc4") or base.endswith(".nc"):
            inferred = infer_original_path_from_sub_filename(base, dataset_dir)
            if inferred:
                hyrax_urls.append(HYRAX_BASE + inferred)
                continue

    # 去重保序
    seen = set()
    uniq = []
    for h in hyrax_urls:
        if h not in seen:
            uniq.append(h)
            seen.add(h)

    if not uniq:
        raise RuntimeError("解析不到 hyrax urls：检查 LINKLIST_PATH 是否指向正确的 linkslist txt。")

    return uniq


# ---------- 提取 mean/max/nearest ----------
def norm_lon_0_360(lon):
    lon = float(lon)
    return lon % 360.0

def roi_slice(ds, lat0, lon0, win):
    lon0 = norm_lon_0_360(lon0)
    latmin, latmax = lat0 - win, lat0 + win
    lonmin, lonmax = lon0 - win, lon0 + win
    if lonmin < 0: lonmin += 360
    if lonmax >= 360: lonmax -= 360

    d2 = ds.sel(lat=slice(latmin, latmax))
    if lonmin <= lonmax:
        d2 = d2.sel(lon=slice(lonmin, lonmax))
    else:
        a = d2.sel(lon=slice(lonmin, 359.999))
        b = d2.sel(lon=slice(0.0, lonmax))
        d2 = xr.concat([a, b], dim="lon")
    return d2

def roi_mean(da, lat):
    w = np.cos(np.deg2rad(lat))
    return da.weighted(w).mean(dim=("lat","lon"), skipna=True)

def date_from_url(url: str):
    m = re.search(r"\.(\d{8})\.nc4", url)
    if not m:
        return None
    return pd.to_datetime(m.group(1), format="%Y%m%d")

def extract_one(url):
    ds = xr.open_dataset(url, engine="netcdf4", chunks={})

    t_utc = pd.to_datetime(ds["time"].values)
    t_loc = t_utc + pd.Timedelta(hours=8)

    out = {"datetime_utc": t_utc, "datetime_local": t_loc}

    d2 = roi_slice(ds, CITY_LAT, CITY_LON, WINDOW_DEG)
    nearest = ds.sel(lat=CITY_LAT, lon=norm_lon_0_360(CITY_LON), method="nearest")

    for v in VARS:
        if v not in ds.variables:
            continue
        da_roi = d2[v]
        if set(["lat","lon"]).issubset(da_roi.dims):
            out[f"{v}_mean"] = roi_mean(da_roi, d2["lat"]).values.reshape(-1)
            out[f"{v}_max"] = da_roi.max(dim=("lat","lon"), skipna=True).values.reshape(-1)
        else:
            out[f"{v}_mean"] = da_roi.values.reshape(-1)
            out[f"{v}_max"] = da_roi.values.reshape(-1)
        out[f"{v}_nearest"] = nearest[v].values.reshape(-1)

    ds.close()
    return pd.DataFrame(out)


def main():
    ensure_earthdata_auth_files(EARTHDATA_USERNAME, EARTHDATA_PASSWORD)

    hyrax_urls = parse_linkslist_to_hyrax_urls(LINKLIST_PATH, DEFAULT_DATASET_DIR)
    print(f"hyrax_urls: {len(hyrax_urls)}")

    start_dt = pd.to_datetime(START)
    end_dt = pd.to_datetime(END)

    urls_sub = []
    for u in hyrax_urls:
        d = date_from_url(u)
        if d is not None and (d >= start_dt) and (d <= end_dt):
            urls_sub.append(u)

    print(f"urls_sub in window: {len(urls_sub)}")

    dfs = [extract_one(u) for u in urls_sub]
    ts = pd.concat(dfs, ignore_index=True).sort_values("datetime_local")
    ts["date"] = pd.to_datetime(ts["datetime_local"]).dt.date

    # unit convert: DUSMASS kg/m3 -> ug/m3
    for c in [c for c in ts.columns if c.startswith("DUSMASS_")]:
        ts[c] = ts[c].astype(float) * 1e9

    daily = ts.groupby("date").agg({
        "DUSMASS_mean": "max",
        "DUSMASS_max": "max",
        "DUSMASS_nearest": "max",
        "DUCMASS_mean": "max",
        "DUEXTTAU_mean": "max",
    }).reset_index()

    print("\nDaily comparison (max within day):")
    print(daily.to_string(index=False))


if __name__ == "__main__":
    main()


hyrax_urls: 366
urls_sub in window: 9


OSError: [Errno -90] NetCDF: file not found: 'https://goldsmr4.gesdisc.eosdis.nasa.gov/opendap/hyrax/data/MERRA2/M2T1NXAER.5.12.4/2021/03/MERRA2_400.tavg1_2d_aer_Nx.20210313.nc4'